In [1]:
import logging
import time
from pathlib import Path
from collections import deque
from typing import Dict, Any
import numpy as np

import gymnasium as gym
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
import src.gymnasium_envs.convex_optimization_env

PROJECT_ROOT = Path().resolve().parent.parent

log_dir = PROJECT_ROOT / "logs"
model_dir = PROJECT_ROOT / "models"

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"CUDA Version: {torch.version.cuda}")

2026-06-03 16:33:31 [INFO] Using device: cuda
2026-06-03 16:33:31 [INFO] GPU: NVIDIA GeForce GTX 1650
2026-06-03 16:33:31 [INFO] CUDA Version: 11.8


Adding status logging

In [3]:
class StatusCallback(BaseCallback):
    def __init__(self, window_size=1000):
        super().__init__()
        self.window = deque(maxlen=window_size)
        self.steps_to_converge = deque(maxlen=window_size)

    def _on_step(self):
        infos = self.locals["infos"]
        for info in infos:
            status = info.get("status")
            if status == "converged":
                self.window.append(1)
                self.steps_to_converge.append(info["iteration"])
            elif status == "diverged":
                self.window.append(-1)

        if len(self.window) > 0:
            self.logger.record("custom/converge_rate",
                self.window.count(1) / len(self.window))
            self.logger.record("custom/diverge_rate",
                self.window.count(-1) / len(self.window))

        if len(self.steps_to_converge) > 0:
            self.logger.record("custom/mean_steps_to_converge",
                np.mean(self.steps_to_converge))  # чем меньше тем лучше

        return True

class BestConvergeCallback(BaseCallback):
    def __init__(self, vec_normalize_env, model_save_path, vec_normalize_save_path, verbose=1):
        super().__init__(verbose)
        self.vec_normalize_env = vec_normalize_env
        self.model_save_path = model_save_path
        self.vec_normalize_save_path = vec_normalize_save_path
        self.best_converge_rate = -np.inf

    def _on_step(self) -> bool:
        converge_rate = self.logger.name_to_value.get("custom/converge_rate", None)

        if converge_rate is not None and converge_rate > self.best_converge_rate:
            self.best_converge_rate = converge_rate
            self.model.save(self.model_save_path)
            self.vec_normalize_env.save(self.vec_normalize_save_path)
            if self.verbose:
                logger.info(f"New best converge_rate: {converge_rate:.4f}")
        return True

Let's create a function for training

In [4]:
def train_model(config: Dict[str, Any], log_dir: str = "logs", model_dir: str = "models", add_time_penalty : bool = False, name_prefix = ""):
    in_features_name = "any" if config["in_features"] is None else config["in_features"]

    in_features_name = name_prefix + in_features_name

    logger.info(f"Starting training for {in_features_name}D task. Timesteps: {config['timesteps']}, Envs: {config['n_envs']}")

    start_time    = time.time()
    model_path    = Path(f"{model_dir}/{in_features_name}d_convex")
    best_model_path = Path(f"{model_dir}/{in_features_name}d_convex_best")
    stats_path    = Path(f"{model_dir}/{in_features_name}d_convex_vec_normalize.pkl")

    model_path.parent.mkdir(parents=True, exist_ok=True)
    best_model_path.mkdir(parents=True, exist_ok=True)

    vec_env = None
    try:
        vec_env = make_vec_env(
            "convex_optimization_env/ConvexOptimization-v1",
            n_envs=config["n_envs"],
            env_kwargs={"in_features": config["in_features"],
                        "add_time_penalty" : add_time_penalty}
        )
        vec_env = VecNormalize(vec_env, norm_obs=False, norm_reward=True)

        best_converge_callback = BestConvergeCallback(
            vec_normalize_env=vec_env,
            model_save_path=str(best_model_path / "best_model"),
            vec_normalize_save_path=str(best_model_path / "best_vec_normalize.pkl"),
            verbose=1
        )

        model = PPO(
            "MultiInputPolicy",
            vec_env,
            verbose=1,
            tensorboard_log=f"{log_dir}/{in_features_name}d/",
            n_steps=config["n_steps"],
            n_epochs=config["n_epochs"],
            batch_size=config["batch_size"],
            learning_rate=config["learning_rate"],
            gamma=config["gamma"],
            clip_range=config["clip_range"],
            ent_coef=config["ent_coef"],
            policy_kwargs=config["policy_kwargs"]
        )

        model.learn(
            total_timesteps=config["timesteps"],
            tb_log_name=f"convex_{in_features_name}d",
            callback=CallbackList([
                StatusCallback(),
                best_converge_callback
            ])
        )

        # Сохраняем финальную модель отдельно
        model.save(str(model_path))
        vec_env.save(str(stats_path))

        elapsed_time = (time.time() - start_time) / 60
        logger.info(f"Training for {in_features_name}D completed. Duration: {elapsed_time:.2f} min.")
        logger.info(f"Final model saved to: {model_path}")
        logger.info(f"Best model saved to: {best_model_path / 'best_model'}")

    except Exception as e:
        logger.error(f"Error during training for {in_features_name}D: {str(e)}", exc_info=True)
    finally:
        if vec_env is not None:
            vec_env.close()

In [5]:
config_base = {
    "in_features"   : None,
    "timesteps"     : 5_000_000, 
    "n_envs"        : 32,
    "n_steps"       : 2048,
    "n_epochs"      : 10, 
    "batch_size"    : 256, 
    "learning_rate" : 3e-4,
    "gamma"         : 0.99,
    "clip_range"    : 0.2,
    "ent_coef"      : 0.01,
    "policy_kwargs" : {
        "net_arch": dict(pi=[64, 64], vf=[64, 64])
    }
}

Let's train a 2D model

In [ ]:
config = config_base
config["in_features"] = 2

train_model(config, log_dir, model_dir)

In [ ]:
config = config_base
config["in_features"] = 5

train_model(config, log_dir, model_dir)

In [ ]:
config = config_base
config["in_features"] = 10

train_model(config, log_dir, model_dir)

In [ ]:
config = config_base
config["in_features"] = 100

train_model(config, log_dir, model_dir)

In [6]:
config = config_base
config["in_features"] = None

train_model(config, log_dir, model_dir)

2026-06-03 16:34:18 [INFO] Starting training for anyD task. Timesteps: 5000000, Envs: 32
c:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\venv\Lib\site-packages\gymnasium\envs\registration.py:728: UserWarning: WARN: The environment is being initialised with render_mode='rgb_array' that is not in the possible render_modes (['ansi']).
  logger.warn(


Using cuda device
Logging to C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\logs/anyd/convex_anyd_3


2026-06-03 16:34:22 [INFO] New best converge_rate: 0.0000
c:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\venv\Lib\site-packages\numpy\linalg\_linalg.py:2767: RuntimeWarning: overflow encountered in dot
  sqnorm = x.dot(x)
C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\src\optimization\optimization_functions\convex_function.py:87: RuntimeWarning: overflow encountered in matmul
  return float(x.T @ self._A @ x + self._b @ x + self._c)
C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\src\gymnasium_envs\convex_optimization_env\envs\convex_optimization_v1.py:323: RuntimeWarning: overflow encountered in dot
  cos_sim = np.dot(grad_unit, prev_grad_unit)
C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\src\gymnasium_envs\convex_optimization_env\envs\convex_optimization_v1.py:110: RuntimeWarning: overflow encountered in multiply
  self._curr_x = prev_x - lr * self._curr_grad
C:\Users\

---------------------------------
| custom/            |          |
|    converge_rate   | 0        |
|    diverge_rate    | 1        |
| rollout/           |          |
|    ep_len_mean     | 641      |
|    ep_rew_mean     | -228     |
| time/              |          |
|    fps             | 5190     |
|    iterations      | 1        |
|    time_elapsed    | 12       |
|    total_timesteps | 65536    |
---------------------------------
-----------------------------------------
| custom/                 |             |
|    converge_rate        | 0           |
|    diverge_rate         | 1           |
| rollout/                |             |
|    ep_len_mean          | 652         |
|    ep_rew_mean          | -225        |
| time/                   |             |
|    fps                  | 2896        |
|    iterations           | 2           |
|    time_elapsed         | 45          |
|    total_timesteps      | 131072      |
| train/                  |             |
|    approx_

2026-06-03 16:43:18 [INFO] New best converge_rate: 0.0026


...

2026-06-03 17:18:25 [INFO] New best converge_rate: 0.3069
2026-06-03 17:18:27 [INFO] New best converge_rate: 0.3079
2026-06-03 17:18:27 [INFO] New best converge_rate: 0.3089
2026-06-03 17:18:34 [INFO] New best converge_rate: 0.3099


-------------------------------------------
| custom/                   |             |
|    converge_rate          | 0.31        |
|    diverge_rate           | 0.69        |
|    mean_steps_to_converge | 377         |
| rollout/                  |             |
|    ep_len_mean            | 968         |
|    ep_rew_mean            | 7.42        |
| time/                     |             |
|    fps                    | 1873        |
|    iterations             | 76          |
|    time_elapsed           | 2658        |
|    total_timesteps        | 4980736     |
| train/                    |             |
|    approx_kl              | 0.003827816 |
|    clip_fraction          | 0.0308      |
|    clip_range             | 0.2         |
|    entropy_loss           | 0.258       |
|    explained_variance     | 0.69        |
|    learning_rate          | 0.0003      |
|    loss                   | 0.000124    |
|    n_updates              | 750         |
|    policy_gradient_loss   | -0

2026-06-03 17:19:08 [INFO] New best converge_rate: 0.3104
2026-06-03 17:19:15 [INFO] New best converge_rate: 0.3114


-------------------------------------------
| custom/                   |             |
|    converge_rate          | 0.311       |
|    diverge_rate           | 0.689       |
|    mean_steps_to_converge | 378         |
| rollout/                  |             |
|    ep_len_mean            | 985         |
|    ep_rew_mean            | 6.96        |
| time/                     |             |
|    fps                    | 1871        |
|    iterations             | 77          |
|    time_elapsed           | 2696        |
|    total_timesteps        | 5046272     |
| train/                    |             |
|    approx_kl              | 0.003323702 |
|    clip_fraction          | 0.0329      |
|    clip_range             | 0.2         |
|    entropy_loss           | 0.271       |
|    explained_variance     | 0.699       |
|    learning_rate          | 0.0003      |
|    loss                   | 0.0109      |
|    n_updates              | 760         |
|    policy_gradient_loss   | -0

2026-06-03 17:19:39 [INFO] Training for anyD completed. Duration: 45.35 min.
2026-06-03 17:19:39 [INFO] Final model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\anyd_convex
2026-06-03 17:19:39 [INFO] Best model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\anyd_convex_best\best_model


In [7]:
config = config_base
config["in_features"] = None

train_model(config, log_dir, model_dir, add_time_penalty = True, name_prefix="time_penalty_")

2026-06-03 17:19:39 [INFO] Starting training for time_penalty_anyD task. Timesteps: 5000000, Envs: 32


Using cuda device
Logging to C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\logs/time_penalty_anyd/convex_time_penalty_anyd_1


c:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\venv\Lib\site-packages\gymnasium\envs\registration.py:728: UserWarning: WARN: The environment is being initialised with render_mode='rgb_array' that is not in the possible render_modes (['ansi']).
  logger.warn(
2026-06-03 17:19:39 [INFO] New best converge_rate: 0.0000


---------------------------------
| custom/            |          |
|    converge_rate   | 0        |
|    diverge_rate    | 1        |
| rollout/           |          |
|    ep_len_mean     | 641      |
|    ep_rew_mean     | -263     |
| time/              |          |
|    fps             | 4447     |
|    iterations      | 1        |
|    time_elapsed    | 14       |
|    total_timesteps | 65536    |
---------------------------------
-----------------------------------------
| custom/                 |             |
|    converge_rate        | 0           |
|    diverge_rate         | 1           |
| rollout/                |             |
|    ep_len_mean          | 622         |
|    ep_rew_mean          | -250        |
| time/                   |             |
|    fps                  | 2561        |
|    iterations           | 2           |
|    time_elapsed         | 51          |
|    total_timesteps      | 131072      |
| train/                  |             |
|    approx_

2026-06-03 17:30:17 [INFO] New best converge_rate: 0.0028


...

2026-06-03 18:09:53 [INFO] New best converge_rate: 0.2705


--------------------------------------------
| custom/                   |              |
|    converge_rate          | 0.27         |
|    diverge_rate           | 0.73         |
|    mean_steps_to_converge | 464          |
| rollout/                  |              |
|    ep_len_mean            | 992          |
|    ep_rew_mean            | -42          |
| time/                     |              |
|    fps                    | 1648         |
|    iterations             | 76           |
|    time_elapsed           | 3021         |
|    total_timesteps        | 4980736      |
| train/                    |              |
|    approx_kl              | 0.0027665019 |
|    clip_fraction          | 0.0232       |
|    clip_range             | 0.2          |
|    entropy_loss           | -0.08        |
|    explained_variance     | 0.934        |
|    learning_rate          | 0.0003       |
|    loss                   | -0.00643     |
|    n_updates              | 750          |
|    polic

2026-06-03 18:10:23 [INFO] New best converge_rate: 0.2717
2026-06-03 18:10:30 [INFO] New best converge_rate: 0.2729
2026-06-03 18:10:35 [INFO] New best converge_rate: 0.2741


--------------------------------------------
| custom/                   |              |
|    converge_rate          | 0.274        |
|    diverge_rate           | 0.726        |
|    mean_steps_to_converge | 459          |
| rollout/                  |              |
|    ep_len_mean            | 975          |
|    ep_rew_mean            | -40.6        |
| time/                     |              |
|    fps                    | 1649         |
|    iterations             | 77           |
|    time_elapsed           | 3060         |
|    total_timesteps        | 5046272      |
| train/                    |              |
|    approx_kl              | 0.0021875156 |
|    clip_fraction          | 0.0193       |
|    clip_range             | 0.2          |
|    entropy_loss           | -0.0835      |
|    explained_variance     | 0.92         |
|    learning_rate          | 0.0003       |
|    loss                   | -0.000862    |
|    n_updates              | 760          |
|    polic

2026-06-03 18:11:01 [INFO] Training for time_penalty_anyD completed. Duration: 51.37 min.
2026-06-03 18:11:01 [INFO] Final model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\time_penalty_anyd_convex
2026-06-03 18:11:01 [INFO] Best model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\time_penalty_anyd_convex_best\best_model
